In [6]:
%pip install pgmpy

  Obtaining dependency information for pgmpy from https://files.pythonhosted.org/packages/9c/01/a3aaac951b59bb4fe3320c263fbd1d7e4d451843b4bee4246eae4fa86688/pgmpy-1.1.0-py3-none-any.whl.metadata
  Using cached pgmpy-1.1.0-py3-none-any.whl.metadata (13 kB)
  Obtaining dependency information for huggingface_hub>=0.23 from https://files.pythonhosted.org/packages/7e/2b/ef03ddb96bd1123503c2bd6932001020292deea649e9bf4caa2cb65a85bf/huggingface_hub-1.12.0-py3-none-any.whl.metadata
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
Using cached pgmpy-1.1.0-py3-none-any.whl (2.4 MB)
Using cached huggingface_hub-1.12.0-py3-none-any.whl (646 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Note: you may ne


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# 2. Import
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# 3. Load Dataset
data = pd.read_csv("Datasets/7_asia_data.csv")

# Rename columns to match the network structure
data.columns = ['Asia', 'Tuberculosis', 'Smoking', 'Dyspnea', 'LungCancer', 'Bronchitis', 'VisitToAsia', 'XRayResult']

# 4. Define Network Structure
model = DiscreteBayesianNetwork([
    ('Smoking', 'LungCancer'),
    ('LungCancer', 'Dyspnea'),
    ('Tuberculosis', 'Dyspnea'),
    ('Asia', 'Tuberculosis'),
    ('Smoking', 'Bronchitis'),
    ('Asia', 'VisitToAsia')
])

# 5. Train Model
model.fit(data)

# 6. Inference
infer = VariableElimination(model)

# 7. Posterior Probability
q = infer.query(variables=['LungCancer'], evidence={'Smoking':1})

print(q)

+---------------+-------------------+
| LungCancer    |   phi(LungCancer) |
+===============+===================+
| LungCancer(1) |            0.9862 |
+---------------+-------------------+
| LungCancer(2) |            0.0138 |
+---------------+-------------------+


In [12]:
# Additional Inference Tests

# Test 1: Probability of Tuberculosis given Asia
q1 = infer.query(variables=['Tuberculosis'], evidence={'Asia':1})
print("P(Tuberculosis | Asia=1):")
print(q1)

# Test 2: Probability of Dyspnea given LungCancer
q2 = infer.query(variables=['Dyspnea'], evidence={'LungCancer':1})
print("\nP(Dyspnea | LungCancer=1):")
print(q2)

# Test 3: Probability of LungCancer given Smoking and Asia
q3 = infer.query(variables=['LungCancer'], evidence={'Smoking':1, 'Asia':1})
print("\nP(LungCancer | Smoking=1, Asia=1):")
print(q3)

P(Tuberculosis | Asia=1):
+-----------------+---------------------+
| Tuberculosis    |   phi(Tuberculosis) |
+=================+=====================+
| Tuberculosis(1) |              0.9929 |
+-----------------+---------------------+
| Tuberculosis(2) |              0.0071 |
+-----------------+---------------------+

P(Dyspnea | LungCancer=1):
+------------+----------------+
| Dyspnea    |   phi(Dyspnea) |
+============+================+
| Dyspnea(1) |         0.9913 |
+------------+----------------+
| Dyspnea(2) |         0.0087 |
+------------+----------------+

P(LungCancer | Smoking=1, Asia=1):
+---------------+-------------------+
| LungCancer    |   phi(LungCancer) |
+===============+===================+
| LungCancer(1) |            0.9862 |
+---------------+-------------------+
| LungCancer(2) |            0.0138 |
+---------------+-------------------+
